# Knowledge Graph Pipeline1: Proof-of-Concept (without ODKE+)

This notebook implements a pipeline for financial knowledge extraction using FNSPID dataset.

In [1]:
!pip install -q neo4j beautifulsoup4 requests google-genai python-dotenv google langchain_neo4j judges

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
genai 2.1.0 requires openai<0.28.0,>=0.27.0, but you have openai 2.15.0 which is incompatible.


In [ ]:
from huggingface_hub import hf_hub_download
import pandas as pd
import os
from pydantic import BaseModel, Field, ValidationError
from typing import Optional, List, Type, TypeVar, Generic
import enum
from datetime import datetime
import requests
from google import genai
from google.genai import types
import json
from neo4j import GraphDatabase, Session, Transaction
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph

In [ ]:
def join_string(item):
    Date, Article_title, Stock_symbol, Url, Lexrank_summary = item
    final_string = ""

    # Check if column has a unique value
    if pd.notna(Date):
        final_string += f"Date: {Date}"

    if pd.notna(Article_title):
        if final_string:
            final_string += " | "
        final_string += f"Article Title: {Article_title}"

    if pd.notna(Stock_symbol):
        if final_string:
            final_string += " | "
        final_string += f"Stock Symbol: {Stock_symbol}"

    if pd.notna(Url):
        if final_string:
            final_string += " | "
        final_string += f"Url: {Stock_symbol}"

    if pd.notna(Lexrank_summary):
        if final_string:
            final_string += " | "
        final_string += f"Summary: {Lexrank_summary}"

    return final_string

def filtered_text(file_path):
    df = pd.read_csv(
        file_path,
        dtype=str,
        low_memory=False,
        encoding='utf-8',
        on_bad_lines='skip'  # Skip problematic lines
    )

    # # DataFrame processing
    # df['Date'] = pd.to_datetime(df['Date'])
    # date_time = input("Date time to filter (e.g., 2023-12-01 00:00:00+00:00): ")
    # ticker_symbol = input("Ticker symbol to filter (e.g., AAPL, AMZN, GOOGL): ")
    # df_filtered = df[(df['Date'].dt.date == pd.to_datetime(date_time).date()) & (df['Stock_symbol'] == ticker_symbol)]

    # # Create the 'information' column
    # df_filtered['Information'] = df_filtered[
    #     ['Date', 'Article_title', 'Stock_symbol','Lexrank_summary']
    # ].apply(join_string, axis=1)

    # # Group all information and return as text
    # df_grouped = df_filtered.groupby('Date')['Information'].apply(lambda x: '\n'.join(x)).reset_index()
    # sample = df_grouped.head().loc[0, 0]['Information']

    # return sample

In [ ]:
def extract_entities_and_relationship(text):
    prompt = f"""
    # Role
    You are an expert Financial Data Analyst and Knowledge Graph Engineer. Your task is to analyze financial news summaries from Nasdaq and extract a structured knowledge graph in JSON format.

    # Task
    You will be provided with a raw text block containing metadata (Date, Title, Ticker, URL) and a Summary. You must extract semantic triplets representing the relationships between entities mentioned in the text.

    # Ontology (Strict Compliance Required)
    You must strictly use ONLY the following Entity Types and Relationship Types. Do not invent new types.

    ## 1. Entity Types (24 Categories)
    **Core Business Entities**
    - ORG: Filing Company (the main subject of the news/ticker)
    - COMP: External companies (competitors, suppliers, customers, partners)
    - SEGMENT: Internal business divisions (e.g., Cloud segment)
    - PERSON: Key individuals (Executives, Board members)

    **Geographic & Regulatory**
    - GPE: Geographic entities (Countries, cities)
    - ORG_GOV: Government bodies (e.g., US Gov)
    - ORG_REG: Regulatory bodies (SEC, Fed, ECB)

    **Financial & Market**
    - FIN_INST: Financial instruments (Stocks, bonds, options)
    - FIN_MARKET: Market indices (S&P 500, Nasdaq)
    - FIN_METRIC: Financial metrics (Revenue, EPS, Share Price)
    - ECON_IND: Economic indicators (Inflation, GDP)

    **Products & Operations**
    - PRODUCT: Products or services (iPhone, AWS)
    - CONCEPT: Abstract concepts (AI, Digital Transformation)
    - RAW_MATERIAL: Essential materials (Lithium, Oil)
    - LOGISTICS: Supply chain entities (Ports, Shipping lanes)

    **Risk & Compliance**
    - RISK_FACTOR: Documented risks (Recession risk, Cyber attacks)
    - LITIGATION: Legal disputes, lawsuits
    - REGULATORY_REQUIREMENT: Specific regulations (GDPR, Basel III)
    - ACCOUNTING_POLICY: Policies (Revenue recognition)

    **Strategic & ESG**
    - EVENT: Material events (Earnings call, M&A, Pandemic)
    - SECTOR: Industries (Technology, Healthcare)
    - ESG_TOPIC: ESG themes (Carbon, DEI)
    - MACRO_CONDITION: Economic trends (Recession, Labor shortage)
    - COMMENTARY: Management statements/guidance

    ## 2. Relationship Types (27 Categories)
    **Ownership & Control**
    - Has_Stake_In
    - Regulates
    - Operates_In

    **Business Activities**
    - Announces
    - Introduces
    - Produces
    - Invests_In
    - Partners_With
    - Supplies

    **Financial Impact**
    - Impacts
    - Positively_Impacts
    - Negatively_Impacts
    - Increases
    - Decreases
    - Affects_Stock

    **Risk & Events**
    - Involved_In
    - Impacted_By
    - Faces
    - Depends_On

    **Market & Reporting**
    - Discloses
    - Guides_On
    - Complies_With
    - Subject_To

    **Additional Relations**
    - Related_To
    - Member_Of
    - Causes_Shortage_Of
    - Stock_Decline_Due_To
    - Stock_Rise_Due_To
    - Market_Reacts_To

    # Extraction Rules
    1. **Metadata Extraction:** Parse the input header to extract the global `date`, `ticker`, and `source_url`. Apply these to every triplet found in that text.
    2. **Entity Resolution:**
    - If the main ticker (e.g., AAPL) is mentioned, label it as `ORG`.
    - Other companies mentioned (e.g., McDonald's in an Apple article) should be labeled `COMP`.
    3. **Granularity:** Extract specific named entities from the text (e.g., "Apple Inc." instead of just "Company").
    4. **Source Text:** You must include the exact sentence or phrase where the relationship was found in the `source_text` field.
    5. **Preserve ALL numerical figures and dates exactly.**
    6. **Keep ALL dates in their original format.**

    # Output Format
    Return a single JSON list of objects. Do not include markdown formatting (like ```json) outside of the list.

    JSON Structure:
    [
        {{
            "triplet_id": "UUID or Sequence (optional)",
            "entity": {{
                "name": "Extracted Head Entity Name",
                "entity_type": "One of the 24 entity types (ORG, COMP, PERSON, etc.)"
            }},
            "relationship": {{
                "relationship": "One of the 27 relationship types (Produces, Increases, etc.)",
                "evidence": "Evidence text snippet supporting the relationship"
            }},
            "target": {{
                "name": "Extracted Tail Entity Name",
                "entity_type": "One of the 24 entity types (PRODUCT, FIN_METRIC, etc.)"
            }},
            "temporal_info": {{
                "date": "YYYY-MM-DD",
                "extraction_type": "extracted or default"
            }},
            "chunk_text": "Full text context surrounding the triplet (the exact sentence/phrase where relationship was found)",
            "chunk_id": null,
            "page_id": null
        }}
    ]

    Note: For compatibility with existing code, also include these flat fields at the root level:
    - "date": "YYYY-MM-DD" (from metadata)
    - "ticker": "Symbol from metadata"
    - "source_url": "URL from metadata"

    # Few-Shot Example

    **Input:**
    Date: 2023-12-01 | Article Title: Best Blue Chip Stocks? | Stock Symbol: AAPL | Url: https://nasdaq.com/article/123
    Summary: Apple Inc. (AAPL) is a multinational technology company that specializes in consumer electronics. Recently, shares of AAPL stock have gained by 9.29% due to holiday sales.

    **Output:**
    [
        {{
            "triplet_id": "1",
            "entity": {{
                "name": "Apple Inc.",
                "entity_type": "ORG"
            }},
            "relationship": {{
                "relationship": "Produces",
                "evidence": "Apple Inc. (AAPL) is a multinational technology company that specializes in consumer electronics."
            }},
            "target": {{
                "name": "consumer electronics",
                "entity_type": "PRODUCT"
            }},
            "temporal_info": {{
                "date": "2023-12-01",
                "extraction_type": "default"
            }},
            "chunk_text": "Apple Inc. (AAPL) is a multinational technology company that specializes in consumer electronics.",
            "chunk_id": null,
            "page_id": null,
            "date": "2023-12-01",
            "ticker": "AAPL",
            "source_url": "https://nasdaq.com/article/123"
        }},
        {{
            "triplet_id": "2",
            "entity": {{
                "name": "AAPL",
                "entity_type": "ORG"
            }},
            "relationship": {{
                "relationship": "Increases",
                "evidence": "shares of AAPL stock have gained by 9.29%"
            }},
            "target": {{
                "name": "Share Price",
                "entity_type": "FIN_METRIC"
            }},
            "temporal_info": {{
                "date": "2023-12-01",
                "extraction_type": "default"
            }},
            "chunk_text": "shares of AAPL stock have gained by 9.29%",
            "chunk_id": null,
            "page_id": null,
            "date": "2023-12-01",
            "ticker": "AAPL",
            "source_url": "https://nasdaq.com/article/123"
        }},
        {{
            "triplet_id": "3",
            "entity": {{
                "name": "AAPL",
                "entity_type": "ORG"
            }},
            "relationship": {{
                "relationship": "Stock_Rise_Due_To",
                "evidence": "shares of AAPL stock have gained by 9.29% due to holiday sales"
            }},
            "target": {{
                "name": "Holiday Sales",
                "entity_type": "EVENT"
            }},
            "temporal_info": {{
                "date": "2023-12-01",
                "extraction_type": "default"
            }},
            "chunk_text": "shares of AAPL stock have gained by 9.29% due to holiday sales.",
            "chunk_id": null,
            "page_id": null,
            "date": "2023-12-01",
            "ticker": "AAPL",
            "source_url": "https://nasdaq.com/article/123"
        }}
    ]

    # Input Text for Processing
    {text}
    """
    load_dotenv()
    client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.1,
            response_mime_type="application/json"
        )
    )

    result = json.loads(response.text)
    return result

In [14]:
def clear_kg(kg):
    with kg._driver.session() as session:
        session.run("MATCH (n) DETACH DELETE n")

def add_relationship_to_neo4j(kg, triplets):
    """
    Add relationships to Neo4j graph.
    Handles both nested FinancialTripletModel structure and flat structure for backward compatibility.
    """
    with kg._driver.session() as session:
        for triplet in triplets:
            # Handle nested FinancialTripletModel structure
            if isinstance(triplet.get('entity'), dict):
                # Nested structure (FinancialTripletModel format)
                entity_name = triplet['entity']['name']
                entity_type = triplet['entity']['entity_type']
                rel_type = triplet['relationship']['relationship']
                target_name = triplet['target']['name']
                target_type = triplet['target']['entity_type']
                source_text = triplet.get('chunk_text', triplet['relationship'].get('evidence', ''))
            else:
                # Flat structure (backward compatibility)
                entity_name = triplet['entity']
                entity_type = triplet['entity_type']
                rel_type = triplet['relationship']
                target_name = triplet['target']
                target_type = triplet['target_type']
                source_text = triplet.get('source_text', '')

            query = f"MERGE (a:Entity {{name: $source}}) " \
                    f"ON CREATE SET a.type = $source_type " \
                    f"MERGE (b:Entity {{name: $target}}) " \
                    f"ON CREATE SET b.type = $target_type " \
                    f"MERGE (a)-[r:{rel_type}]->(b) " \
                    f"SET r.date = $date, " \
                    f"r.ticker = $ticker, " \
                    f"r.source_url = $source_url, " \
                    f"r.source_text = $source_text"
            try:
                session.run(
                    query,
                    source=entity_name,
                    source_type=entity_type,
                    target=target_name,
                    target_type=target_type,
                    date=triplet.get('date', 'N/A'),
                    ticker=triplet.get('ticker', 'N/A'),
                    source_url=triplet.get('source_url', ''),
                    source_text=source_text
                )
            except Exception as e:
                print(f"Error adding relationship: {e}")

In [48]:
# Create dummy questions and answers for testing
test_questions = {
    'questions': [
        'What is the stock price change for AAPL on 2023-12-01?',
        'Which companies are mentioned in relation to AAPL?',
        'What products does Apple produce?',
        'What events impacted AAPL stock?',
        'What is the relationship between AAPL and consumer electronics?'
    ],
    'expected': [
        'AAPL stock increased by 9.29% due to holiday sales.',
        'Apple Inc. is the main company discussed with AAPL ticker.',
        'Apple produces consumer electronics.',
        'Holiday sales caused the stock to rise.',
        'Apple is a technology company that produces consumer electronics.'
    ]
}
test_output = {
    'output': [
        'AAPL stock gained by 9.29% on 2023-12-01.',
        'Apple Inc. and McDonald\'s are mentioned.',
        'Apple produces consumer electronics including iPhone and other devices.',
        'Holiday sales positively impacted AAPL stock causing a rise.',
        'Apple Inc. produces consumer electronics as a multinational technology company.'
    ]
}

dummy_questions_df = pd.DataFrame(test_questions)
dummy_answer_df = pd.DataFrame(test_output)
dummy_questions_df.to_csv('dummy_questions_llm_judge.csv',index=False)
dummy_answer_df.to_csv("dummy_answers_llm_judge.csv",index=False)


In [ ]:
%%writefile llm_judge.py
'''
Implementation of LLM as a judge 

Version: 2026-1-13

Function:
- grader_judge
- classic_classifier
- classifer_judge

'''
from tqdm.auto import tqdm
import pandas as pd
from pandas import DataFrame
from openai import OpenAI
from textwrap import dedent
from judges.base import BaseJudge, Judgment
from judges.graders.correctness import PrometheusAbsoluteCoarseCorrectness
import time
from sklearn.metrics import accuracy_score, f1_score, classification_report, precision_recall_fscore_support


# 
class KnowledgeGraphJudge(BaseJudge):
    def judge(
            self,
            input: str,
            output: str = None,
            expected: str = None,
    ) -> Judgment:
        """    
        Parameters:
        -----------
        input: str
            The input provided to the model to be judged.
        output: str  
            The output generated by the model.
        expected: str
            The expected output for comparison (optional).
        
        Returns:
        --------
        Judgment:
            The evaluation result containing the score and reasoning.
        """
        system_prompt = "You are a financial domain expert"
        user_prompt = dedent(
            f"""
            Evaluate whether the following response is correct base on the provided information

            Original questions: {input}
            Response to evaluate: {output}

            Return a score from 1 to 5 with 1 being the least and 5 being the best
            """
        )
        
        reasoning, score = self._judge(
            user_prompt=user_prompt,
            system_prompt=system_prompt,
        )

        return Judgment(reasoning=reasoning,score=score, score_type="numerical")

def grader_judge(
    questions_path: str,
    model_output_path: str,
    evaluator: dict,
    save_output: bool = True,
    save_name: str = "judge_output.csv"
) -> pd.DataFrame:
    """
    Evaluate model outputs using multiple metrics and save results in a structured format.
    
    Parameters:
    -----------
    questions_path: str
        Path to CSV file containing questions
    model_output_path: str
        Path to CSV file containing model outputs
    evaluator: dict
        Dictionary mapping metric names to BaseJudge instances
    save_output: bool
        Whether to save results to CSV
    save_name: str
        Output CSV filename
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with all questions, outputs, and metric scores
    """
    start_time = time.time()
    
    # Load the questions and outputs once
    questions_df = pd.read_csv(questions_path)
    model_output_df = pd.read_csv(model_output_path)
    combined_df = pd.concat([questions_df, model_output_df], axis=1)
    
    print(f"[INFO] Loaded {len(combined_df)} question-answer pairs")
    
    # Initialize result DataFrame with input data
    results_df = combined_df.copy()
    
    # Evaluate each metric
    for metric_name, model in evaluator.items():
        print(f"[INFO] Model {model.model} is evaluating {metric_name} using {model}...")
        scores = []
        reasons = []
        
        for idx, row in tqdm(combined_df.iterrows(), total=len(combined_df), desc=f"Evaluating {metric_name}"):
            try:
                question = row["questions"]
                output = row["output"]
                expected = row.get("expected", None)
                
                judgment = model.judge(
                    input=question,
                    output=output,
                    expected=expected,
                )
                
                score_val = judgment.score if isinstance(judgment.score, (int, float)) else 0
                reason_val = getattr(judgment, "reasoning", "")
                
            except Exception as e:
                score_val = 0
                reason_val = f"judge error: {e}"
                print(f"[ERROR] Error at index {idx} for {metric_name}: {e}")
            
            scores.append(score_val)
            reasons.append(reason_val)
        
        # Add metric results as columns
        results_df[f"{metric_name}_score"] = scores
        results_df[f"{metric_name}_reasoning"] = reasons
    
    # Calculate average score across all metrics
    score_columns = [col for col in results_df.columns if col.endswith("_score")]
    if score_columns:
        results_df["average_score"] = results_df[score_columns].mean(axis=1)
    
    # Save results
    if save_output:
        results_df.to_csv(save_name, index=False)
        print(f"[INFO] Results saved to {save_name}")
    
    total_time = time.time() - start_time
     
    print_result(
        no_questions=len(questions_df),
        total_time=total_time,
        score_columns=score_columns,
        results_df=results_df
    )

    return results_df

def classifer_judge(
        questions_path:str,
        model_output_path:str,
        judges: dict,
        save_result: True,
        save_name: str ="classifer_judge_results.csv"
):  
    start_time = time.time()
    
    # Load the questions and outputs once
    questions_df = pd.read_csv(questions_path)
    model_output_df = pd.read_csv(model_output_path)
    combined_df = pd.concat([questions_df, model_output_df], axis=1)
    
    print(f"[INFO] Loaded {len(combined_df)} question-answer pairs")
    
    # Initialize result DataFrame with input data
    results_df = combined_df.copy()
    
    # Evaluate each metric
    for metric_name, model in judges.items():
        print(f"[INFO] Model {model.model} is evaluating {metric_name} using {model}...")
        scores = []
        reasons = []
        
        for idx, row in tqdm(combined_df.iterrows(), total=len(combined_df), desc=f"Evaluating {metric_name}"):
            try:
                question = row["questions"]
                output = row["output"]
                expected = row.get("expected", None)
                
                judgment = model.judge(
                    input=question,
                    output=output,
                    expected=expected,
                )
                
                if judgment.score == True:
                    score_val = 1
                else:
                    score_val = 0
                reason_val = getattr(judgment, "reasoning", "")
                
            except Exception as e:
                score_val = 0
                reason_val = f"judge error: {e}"
                print(f"[ERROR] Error at index {idx} for {metric_name}: {e}")
            
            scores.append(score_val)
            reasons.append(reason_val)
        
        # Add metric results as columns
        results_df[f"{metric_name}_score"] = scores
        results_df[f"{metric_name}_reasoning"] = reasons
    
    # Calculate average score across all metrics
    score_columns = [col for col in results_df.columns if col.endswith("_score")]
    if score_columns:
        results_df["average_score"] = results_df[score_columns].mean(axis=1)
    
    # Save results
    if save_result:
        results_df.to_csv(save_name, index=False)
        print(f"[INFO] Results saved to {save_name}")
    
    total_time = time.time() - start_time

    print_result(
        no_questions=len(questions_df),
        total_time=total_time,
        score_columns=score_columns,
        results_df=results_df
    )
    return results_df
    


def goldenset_classifier(
    questions_path: str,
    model_output_path: str,
    label_col: str = "expected",
    pred_col: str = "output",
    save_output: bool = True,
    save_name: str = "classifier_judge.csv",
    pos_label=None,
) -> pd.DataFrame:
    """
    Evaluate classifier outputs against golden labels and report F1.

    Parameters
    ----------
    questions_path : str
        CSV with the golden labels (column `label_col`).
    model_output_path : str
        CSV with model predictions (column `pred_col`).
    label_col : str
        Column name for ground-truth labels in questions CSV.
    pred_col : str
        Column name for predicted labels in model outputs CSV.
    pos_label : optional
        Positive class for binary F1. If None, macro/weighted F1 are reported.
    """
    t0 = time.time()
    gt_df = pd.read_csv(questions_path)
    pred_df = pd.read_csv(model_output_path)

    if label_col not in gt_df.columns:
        raise ValueError(f"Label column '{label_col}' not found in {questions_path}")
    if pred_col not in pred_df.columns:
        raise ValueError(f"Prediction column '{pred_col}' not found in {model_output_path}")

    combined = pd.concat([gt_df.reset_index(drop=True), pred_df.reset_index(drop=True)], axis=1)
    combined.rename(columns={label_col: "gold", pred_col: "pred"}, inplace=True)

    # Per-example correctness
    combined["correct"] = combined["gold"] == combined["pred"]

    # Metrics
    acc = accuracy_score(combined["gold"], combined["pred"])
    macro_f1 = f1_score(combined["gold"], combined["pred"], average="macro")
    weighted_f1 = f1_score(combined["gold"], combined["pred"], average="weighted")
    if pos_label is not None:
        binary_f1 = f1_score(combined["gold"], combined["pred"], average="binary", pos_label=pos_label)
    else:
        binary_f1 = None

    cls_report = classification_report(combined["gold"], combined["pred"], digits=3)

    if save_output:
        combined.to_csv(save_name, index=False)
        print(f"[INFO] Classifier results saved to {save_name}")

    elapsed = time.time() - t0

    # Report
    print(f"\n{'='*70}")
    print("CLASSIFIER EVALUATION REPORT")
    print(f"{'='*70}")
    print(f"Samples: {len(combined)}")
    print(f"Total Time: {elapsed:.2f}s ({elapsed/60:.2f} min)")
    print(f"Accuracy: {acc:.4f}")
    print(f"Macro F1: {macro_f1:.4f}")
    print(f"Weighted F1: {weighted_f1:.4f}")
    if binary_f1 is not None:
        print(f"Binary F1 (pos_label={pos_label}): {binary_f1:.4f}")
    print(f"\nPer-class metrics:\n{cls_report}")
    print(f"{'='*70}\n")

    return combined

def print_result(
        no_questions,
        total_time,
        score_columns,
        results_df,
):
    """
    Utilisites function use to print the results
    """
    # Print detailed results report
    print(f"\n{'='*70}")
    print(f"EVALUATION RESULTS REPORT")
    print(f"{'='*70}")
    print(f"Total Questions Evaluated: {no_questions}")
    print(f"Total Evaluation Time: {total_time:.2f}s ({total_time/60:.2f} minutes)")
    print(f"\n{'-'*70}")
    print(f"METRIC SCORES:")
    print(f"{'-'*70}")

    for col in score_columns:
        metric_name = col.replace("_score", "")
        mean_score = results_df[col].mean()
        std_score = results_df[col].std()
        min_score = results_df[col].min()
        max_score = results_df[col].max()
        print(f"  {metric_name.capitalize():15s} | Mean: {mean_score:.2f} | Std: {std_score:.2f} | Min: {min_score:.2f} | Max: {max_score:.2f}")
    
    if "average_score" in results_df.columns:
        avg_mean = results_df["average_score"].mean()
        avg_std = results_df["average_score"].std()
        print(f"{'-'*70}")
        print(f"  {'Overall Average':15s} | Mean: {avg_mean:.2f} | Std: {avg_std:.2f}")
    
    print(f"{'='*70}\n")

if __name__ == "__main__":
    pass
    # from judges.graders.correctness import PrometheusAbsoluteCoarseCorrectness
    # from judges.graders.relevance import ReliableCIRelevance
    # from judges.classifiers.correctness import PollZeroShotCorrectness
    # # Demo how to use
    # JUDGE_MODEL = "ollama/gpt-oss:20b"
    # TEST_QUESTION = "dummy_questions_llm_judge.csv"
    # TEST_OUTPUT = "dummy_answers_llm_judge.csv"
    # GRADERS = {
    #     "correctness": PrometheusAbsoluteCoarseCorrectness(model=JUDGE_MODEL),
    #     "relevance": ReliableCIRelevance(model=JUDGE_MODEL)
    # }

    # CLASSIFIERS = {
    #     "correctness": PollZeroShotCorrectness(model=JUDGE_MODEL)
    # }
    # # results = grader_judge(
    # #     TEST_QUESTION,
    # #     TEST_OUTPUT,
    # #     GRADERS,
    # #     save_output=True,
    # #     save_name="test_2_metrics_LLM_judge.csv"
    # # )

    # results = classifer_judge(
    #     TEST_QUESTION,
    #     TEST_OUTPUT,
    #     CLASSIFIERS,
    #     True,
    #     "6_sample_classifier.csv"
    # )
    
    # print(f"Results shape: {results.shape}")
    # print(f"Columns: {results.columns.tolist()}")

Overwriting llm_judge.py


In [ ]:
import csv
if __name__ == "__main__":
    # 1. Download dataset from huggingface
    # file_path = hf_hub_download(
    #     repo_id="Zihan1004/FNSPID",
    #     filename="Stock_news/nasdaq_exteral_data.csv",
    #     repo_type="dataset"
    # )
    # sample = filtered_text(file_path)
    #with open('/Users/anhvu/Desktop/Submission/IDM/Assignment3/sample-text', 'r') as f: # hard-coded for POC development
    #    sample = f.read()

    # 2. Load and filter CSV data for AAPL on 2023-12-01 using financial_ontology

    # Load CSV and filter for AAPL on 2023-12-01
    file_path = 'nasdaq_100.csv'
    df = pd.read_csv(
        file_path,
        dtype=str,
        low_memory=False,
        encoding='utf-8',
        on_bad_lines='skip',
        sep=',',
        quotechar='"',
        quoting=csv.QUOTE_MINIMAL
    )

    # Filter for AAPL on 2023-12-01
    df['Date'] = pd.to_datetime(df['Date'])
    target_date = pd.to_datetime('2023-12-01').date()
    df_filtered = df[
        (df['Date'].dt.date == target_date) &
        (df['Stock_symbol'] == 'AAPL')
    ]

    if df_filtered.empty:
        print(f"No data found for AAPL on 2023-12-01")
        triplets = []
    else:
        # Create the 'information' column using join_string function
        df_filtered['Information'] = df_filtered[
            ['Date', 'Article_title', 'Stock_symbol', 'Url', 'Lexrank_summary']
        ].apply(join_string, axis=1)

        # Group all information and return as text
        df_grouped = df_filtered.groupby('Date')['Information'].apply(
            lambda x: '\n'.join(x)
        ).reset_index()

        if len(df_grouped) > 0:
            sample_text = df_grouped.iloc[0]['Information']
            print(f"Extracted text from {len(df_filtered)} articles for AAPL on 2023-12-01")

            # Use LLMs to extract kg entities and relationship
            triplets = extract_entities_and_relationship(sample_text)
            print(f"Extracted {len(triplets)} triplets using financial_ontology structure")
        else:
            print("No grouped data found")
            triplets = []

    # 3. Add relationships to Neo4j Instance
    load_dotenv()
    graph = Neo4jGraph(
        url=userdata.get('NEO4J_URI'),
        username=userdata.get('NEO4J_USERNAME'),
        password=userdata.get('NEO4J_PASSWORD'),
        database=userdata.get('NEO4J_DATABASE')
    )
    clear_kg(graph)
    add_relationship_to_neo4j(graph, triplets)

    # 4. LLMs output from few-shot guiding questions (10 questions)
    
    # 5. Run LLM as a judge on input and output, compare with groundtruth answer done by human

/tmp/ipython-input-1463812298.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['Information'] = df_filtered[


Extracted text from 20 articles for AAPL on 2023-12-01
Extracted 97 triplets using financial_ontology structure


In [1]:
import pandas as pd

answers = pd.read_csv("results.csv")
result_sample = answers[["Result"]].rename(columns={"Result":"output"})
# result_sample
result_sample.to_csv("results_judge_prep.csv",index=False) 

In [2]:
from llm_judge import classifer_judge
from judges.classifiers.correctness import PollMultihopCorrectness

questions_path = "6_samples_questions.csv"
answers_path = "results_judge_prep.csv"
model = "ollama/gpt-oss:20b"

correctness_judge = PollMultihopCorrectness(model=model)

evaluators = {
    "correctness": correctness_judge
}

classifer_judge(questions_path,answers_path,evaluators,True,"result_1_judge.csv")

/home/dang/miniconda3/envs/kg-improved/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[INFO] Loaded 7 question-answer pairs
[INFO] Model ollama/gpt-oss:20b is evaluating correctness using PollMultihopCorrectness()...


Evaluating correctness:  57%|█████▋    | 4/7 [01:08<01:03, 21.15s/it]

[ERROR] Error at index 3 for correctness: <failed_attempts>

<generation number="1">
<exception>
    2 validation errors for Judgment
score
  Field required [type=missing, input_value={'judgments': [{'REASONIN...tent.', 'SCORE': True}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
reasoning
  Field required [type=missing, input_value={'judgments': [{'REASONIN...tent.', 'SCORE': True}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
</exception>
<completion>
    ChatCompletion(id='chatcmpl-833', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "judgments": [\n    {\n      "REASONING": "The provided answer correctly states that George Washington left office on March 4, 1797, which matches the reference date.",\n      "SCORE": true\n    },\n    {\n      "REASONING": "The reference answer does not provide a location; it simply repeats th

Evaluating correctness: 100%|██████████| 7/7 [05:59<00:00, 51.34s/it]

[ERROR] Error at index 6 for correctness: <failed_attempts>

<generation number="1">
<exception>
    2 validation errors for Judgment
score
  Field required [type=missing, input_value={'judgments': [{'score': ...core_type': 'boolean'}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
reasoning
  Field required [type=missing, input_value={'judgments': [{'score': ...core_type': 'boolean'}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
</exception>
<completion>
    ChatCompletion(id='chatcmpl-244', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "judgments": [\n    {\n      "score": true,\n      "reasoning": "The provided answer states that George Washington left office on March 4, 1797, which matches the reference answer.",\n      "score_type": "boolean"\n    },\n    {\n      "score": false,\n      "reasoning": "The provided answer cla

,questions,expected,output,correctness_score,correctness_reasoning,average_score
0,What products does Apple produce?,"Iphone,Consumer electronics, Software, Online ...",Apple Inc. (AAPL) is a multinational technolog...,0,The provided answer lists consumer electronics...,0.0
1,How much did AAPL stock increase?,"""AAPL stock rise by 2.6%""",AAPL stock has to rise by 2.6% from sourceURL:...,0,The evaluation of each provided answer against...,0.0
2,Who are Apple's competitors?,"Alphabet, Amazon, Samsung",Alphabet to compete well with some notable ind...,0,The reference answer lists Apple’s competitors...,0.0
3,What events affected Apple's stock?,Can not be found in current knowledge base,No results found,0,judge error: <failed_attempts>\n\n<generation ...,0.0
4,What products does Apple produce and which eve...,Can not be found in current knowledge base,No results found,0,The user creation endpoint is a standard RESTf...,0.0
5,How many relationships does Apple have in the ...,"Based on the extracted knowledge graph, Apple ...",Company: Apple | TotalRelationships: 18 | Rela...,0,"Judgments for each question: Q1 True, Q2 False...",0.0
6,What happened with Apple in December 2023?,"In December 2023, Apple Inc. (AAPL) was widely...",the company had its best fiscal Q4 ever for iP...,0,judge error: <failed_attempts>\n\n<generation ...,0.0
